# 71_percent_colab_t4_playbook
Notebook ini untuk menjalankan pipeline IndoBERT+BiLSTM di Google Colab T4 dengan konfigurasi yang bisa dipilih sesuai target (stable/f1_boost/safe).

## 0) Clone & Setup
Jalankan dari runtime baru Colab.

In [ ]:
!git clone https://github.com/kzquandary/mbg-sentiment.git
%cd mbg-sentiment
!git checkout main
!python3 -m pip install -r requirements.txt

## 1) Upload Dataset
Upload `dataset_relabel_mbg.csv` ke folder `data/`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pilih dataset_relabel_mbg.csv
!mkdir -p data
!mv dataset_relabel_mbg.csv data/

## 2) Persiapan Data (CPU)
- Split utama 70/30
- Preprocess train/test
- Split internal train_sub/val_sub
- Balancing hanya train_sub

In [ ]:
!python3 src/05_split_data.py --input data/dataset_relabel_mbg.csv --text-col text --label-col Labeling_Sentimen --train-ratio 0.7 --val-ratio 0 --test-ratio 0.3
!python3 src/03_preprocess_text.py --train-input data/train.csv --test-input data/test.csv --text-col text

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/train.csv')
train_sub, val_sub = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Labeling_Sentimen'].astype(str))
train_sub.to_csv('data/train_sub.csv', index=False, encoding='utf-8-sig')
val_sub.to_csv('data/val_sub.csv', index=False, encoding='utf-8-sig')
print('train_sub', len(train_sub), train_sub['Labeling_Sentimen'].value_counts().to_dict())
print('val_sub  ', len(val_sub), val_sub['Labeling_Sentimen'].value_counts().to_dict())

In [ ]:
!python3 src/05b_balance_train.py --input data/train_sub.csv --label-col Labeling_Sentimen --target-mode fixed --target-count 1500 --output data/train_sub_balanced.csv --log-output outputs/train_sub_balance_log.json

## 3) Pilih Konfigurasi Training (JSON)
Gunakan salah satu:
- `src/resources/step7_colab_t4_stable.json`: titik awal paling stabil
- `src/resources/step7_colab_t4_f1_boost.json`: dorong macro F1 (lebih berat)
- `src/resources/step7_colab_t4_safe.json`: fallback jika memori sempit

In [ ]:
CONFIG_PATH = 'src/resources/step7_colab_t4_stable.json'  # ganti ke f1_boost/safe sesuai kebutuhan
print('Using config:', CONFIG_PATH)

## 4) Train IndoBERT + BiLSTM (CUDA)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
# os.environ['HF_TOKEN'] = 'hf_xxx'  # optional

In [ ]:
!python3 src/07_indobert_bilstm.py --train data/train_sub_balanced.csv --val data/val_sub.csv --trial-configs-json {CONFIG_PATH} --max-trials 3

## 5) Evaluasi Test Asli

In [ ]:
!python3 src/09_evaluate.py --test data/test.csv --model-path models/best_indobert_bilstm.pt

In [ ]:
import json, pandas as pd
metrics = json.load(open('outputs/final_metrics.json', encoding='utf-8'))
print(metrics)
report = pd.read_csv('outputs/classification_report.csv')
report

## 6) Catatan Interpretasi
- Gunakan macro F1 untuk menilai kestabilan antar kelas.
- Accuracy bisa tinggi saat kelas Netral dominan, jadi jangan dipakai sendirian.
- Jika macro F1 turun tajam, cek lagi balancing ratio (`target-count`) dan config trial.